# TAE-IA · Module 6 · L12 — Product Photo Generator

**Objetivo:** convertir una foto de producto (celular, sin estudio) en fotos estilo
marketing, con descripción, categoría y narración en audio — detrás de una app Gradio
en vivo.

**Modelos:** CLIP ViT-B/32 · BLIP-large · Stable Diffusion 1.5 (img2img + inpainting)
— visión · Piper TTS `es_MX-claude-high` — audio · Helsinki-NLP/opus-mt-en-es
(traducción) · rembg · Applio/RVC opcional (L22) — apoyo, no obligatorios.

**Secciones:** 1 Setup · 2 Fotos · 3 Modelos · 3b Voz clonada (opcional) ·
4 Contrato de entrada · 5 Pipeline · 6 Prueba · 7 App · 8 Ética · 9 Limpieza.

*Reporte completo (objetivo detallado, cumplimiento de los 3 requisitos, limitaciones,
trabajo futuro) en el documento Word aparte.*

## 1 — Setup

Colab does not ship `ultralytics`, `diffusers` or `gradio`, so install them first — about a
minute. Then: weights go on the runtime disk (fast, roomy, wiped between sessions), while your
images and outputs go on Drive, where they survive.

In [1]:
# LIBRERÍAS DEL PROYECTO (pip)
#   ultralytics   -> USAR PARA: YOLO (detección de objetos, heredado de L11)
#   diffusers     -> USAR PARA: Stable Diffusion (genera las fotos de marketing)
#   transformers  -> USAR PARA: CLIP (clasificación) y BLIP (descripción)
#   accelerate    -> USAR PARA: acelerar la carga e inferencia de los modelos en GPU
#   gradio        -> USAR PARA: la app web en vivo (requisito 3)
#   piper-tts     -> USAR PARA: texto a voz, narra la descripción (requisito 2, audio)
#   rembg         -> USAR PARA: quitar el fondo (apoyo del estilo "fondo negro profesional")
#                    (<2.0.70 evita el choque de Pillow con gradio)
!pip install -q "ultralytics>=8.3,<8.4" "diffusers>=0.31,<0.32" "transformers>=4.46,<4.47" "accelerate>=1.1,<1.2" "gradio>=5.6,<5.7" "piper-tts>=1.8,<1.9" "rembg>=2.0,<2.0.70"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 4.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 25.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 25.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.2/333.2 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.1/57.1 MB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.1/320.1 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.1/34.1 MB 38.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 52.6 MB/s eta 0:00:00
   ━━━━━━━

In [2]:
# LIBRERÍAS
import os, time, random
import numpy as np
import torch
from PIL import Image, ImageDraw, ImageFilter

# MONTAR DRIVE
from google.colab import drive
drive.mount('/content/drive')

# GUARDAR EN DRIVE (entradas y salidas del proyecto)
DRIVE_ROOT = '//content/drive/MyDrive/Colab Notebooks/Module 6/Dr. Francisco J. Rodriguez'
OUTPUT_DIR = f'{DRIVE_ROOT}/L12_output'
INPUT_DIR  = f'{DRIVE_ROOT}/L12_inputs'
for d in (OUTPUT_DIR, INPUT_DIR):
    os.makedirs(d, exist_ok=True)

# MODELOS (DISCO LOCAL, no Drive)
MODEL_CACHE = '/content/models'
os.makedirs(MODEL_CACHE, exist_ok=True)
os.environ['HF_HOME']         = MODEL_CACHE
os.environ['TORCH_HOME']      = MODEL_CACHE
os.environ['YOLO_CONFIG_DIR'] = MODEL_CACHE

# VERIFICAR GPU
if not torch.cuda.is_available():
    raise SystemExit('No GPU. Runtime > Change runtime type > T4 GPU')

# SEMILLA (reproducibilidad)
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# MONITOREO DE VRAM
def vram(tag=''):
    used  = torch.cuda.memory_allocated() / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'  VRAM {used:5.2f} / {total:.1f} GB   {tag}')

print(torch.cuda.get_device_name(0)); vram('empty')

Mounted at /content/drive
Tesla T4
  VRAM  0.00 / 15.6 GB   empty


## 2 — My five product photos

Five product photos live in `L12_inputs/` — a t-shirt, a mug, handmade jewelry, a candle,
and a bag. Kept separate from L11's `inputs/` folder since these need to look like real
product photography, not general COCO-style test images.

In [3]:
# EXTENSIONES DE IMAGEN VÁLIDAS
IMG_EXTS = ('.jpg', '.jpeg', '.png', '.webp', '.bmp')

# SUBIR FOTOS
def upload_inputs():
    from google.colab import files
    for name, data in files.upload().items():
        with open(os.path.join(INPUT_DIR, name), 'wb') as f:
            f.write(data)
        print('saved', name)

# CARGAR FOTOS
def load_inputs(n=5):
    names = [f for f in sorted(os.listdir(INPUT_DIR))
             if f.lower().endswith(IMG_EXTS)][:n]
    if not names:
        raise SystemExit(f'No images in {INPUT_DIR}. Run upload_inputs() first.')
    return {f: Image.open(os.path.join(INPUT_DIR, f)).convert('RGB') for f in names}

# EJECUCIÓN
upload_inputs()          # correr solo si INPUT_DIR está vacío
print(f'{len(load_inputs())} images ready in {INPUT_DIR}')

Saving cadena.jpg to cadena.jpg
Saving Collar.jpeg to Collar.jpeg
Saving cup.jpg to cup.jpg
Saving t-shirt.jpg to t-shirt.jpg
saved cadena.jpg
saved Collar.jpeg
saved cup.jpg
saved t-shirt.jpg
4 images ready in //content/drive/MyDrive/Colab Notebooks/Module 6/Dr. Francisco J. Rodriguez/L12_inputs


## 3 — Load the models

Three models this time, not L11's YOLO/BLIP/SD combo: **CLIP** replaces YOLO here,
because product categories (jewelry, candles, handmade goods) aren't COCO classes —
YOLO was never going to detect "a candle." CLIP's zero-shot classification against a
custom category list fits a catalog of arbitrary product types much better than a
fixed 80-class detector.

In [4]:
# LIBRERÍAS
from huggingface_hub import snapshot_download, hf_hub_download
from ultralytics import YOLO
from transformers import (BlipProcessor, BlipForConditionalGeneration, CLIPProcessor, CLIPModel,
                          MarianMTModel, MarianTokenizer)
from diffusers import StableDiffusionImg2ImgPipeline, StableDiffusionInpaintPipeline, DPMSolverMultistepScheduler
import wave
from piper import PiperVoice, SynthesisConfig

# DESCARGA DE MODELOS (helper)
def fetch(repo, name, patterns):
    """Download only the files we need - most repos ship the same weights 3x."""
    target = os.path.join(MODEL_CACHE, name)
    snapshot_download(repo, local_dir=target, allow_patterns=patterns)
    return target

# YOLO
yolo = YOLO('yolov8n.pt')
print(f'YOLO: {len(yolo.names)} COCO classes')

# CLIP (VISIÓN — clasificación)
clip_dir   = fetch('openai/clip-vit-base-patch32', 'clip-vit-base-patch32',
                   ['*.json', '*.txt', 'pytorch_model.bin'])
clip_proc  = CLIPProcessor.from_pretrained(clip_dir)
clip_model = CLIPModel.from_pretrained(clip_dir).to('cuda').eval()
vram('after CLIP')

# BLIP (VISIÓN — descripción)
blip_dir   = fetch('Salesforce/blip-image-captioning-large', 'blip-large',
                   ['*.json', '*.txt', '*.safetensors'])
blip_proc  = BlipProcessor.from_pretrained(blip_dir)
blip_model = BlipForConditionalGeneration.from_pretrained(
    blip_dir, torch_dtype=torch.float16).to('cuda').eval()
vram('after BLIP')

# STABLE DIFFUSION (VISIÓN — generación)
sd_dir  = fetch('stable-diffusion-v1-5/stable-diffusion-v1-5', 'sd15',
                ['*.json', '*.txt', '*.fp16.safetensors'])
pipe_sd = StableDiffusionImg2ImgPipeline.from_pretrained(
    sd_dir, torch_dtype=torch.float16, variant='fp16').to('cuda')
pipe_sd.scheduler = DPMSolverMultistepScheduler.from_config(pipe_sd.scheduler.config)   # ~20 pasos en vez de ~50
pipe_sd.set_progress_bar_config(disable=True)
vram('all three resident')

# STABLE DIFFUSION INPAINTING (estilo "fondo negro profesional")
sd_inpaint_dir = fetch('stable-diffusion-v1-5/stable-diffusion-inpainting', 'sd15-inpaint',
                       ['*.json', '*.txt', '*.fp16.safetensors'])
pipe_sd_inpaint = StableDiffusionInpaintPipeline.from_pretrained(
    sd_inpaint_dir, torch_dtype=torch.float16, variant='fp16').to('cuda')
pipe_sd_inpaint.scheduler = DPMSolverMultistepScheduler.from_config(pipe_sd_inpaint.scheduler.config)
pipe_sd_inpaint.set_progress_bar_config(disable=True)
vram('all three + SD-inpaint resident')

# PIPER (AUDIO)
PIPER_STEM = 'es/es_MX/claude/high/es_MX-claude-high'
piper_onnx = hf_hub_download('rhasspy/piper-voices', PIPER_STEM + '.onnx', local_dir=f'{MODEL_CACHE}/piper')
hf_hub_download('rhasspy/piper-voices', PIPER_STEM + '.onnx.json', local_dir=f'{MODEL_CACHE}/piper')
piper_voice = PiperVoice.load(piper_onnx)
vram('after CLIP+BLIP+SD+Piper')

# TRADUCTOR EN→ES
MT_NAME  = 'Helsinki-NLP/opus-mt-en-es'
mt_tok   = MarianTokenizer.from_pretrained(MT_NAME)
mt_model = MarianMTModel.from_pretrained(MT_NAME, torch_dtype=torch.float16).to('cuda').eval()
vram('after CLIP+BLIP+SD+Piper+traductor')

# REMBG
from rembg import remove as rembg_remove
rembg_remove(Image.new('RGB', (32, 32), 'white'))
print('rembg listo (modelo de recorte de fondo cacheado)')

# CHECKLIST DE REQUISITOS
print()
print('=== Checklist de requisitos obligatorios ===')
vision_ok = all(n in globals() for n in ('clip_model', 'blip_model', 'pipe_sd'))
audio_ok  = 'piper_voice' in globals()
print(f"{'✅' if vision_ok else '❌'} Requisito 1/3 -- Modelo de visión: CLIP + BLIP + Stable Diffusion cargados")
print(f"{'✅' if audio_ok else '❌'} Requisito 2/3 -- Modelo de audio: Piper TTS ({PIPER_STEM.split('/')[-1]}) cargado")
print('ℹ️  Requisito 3/3 -- App Gradio en vivo: se confirma al correr la Sección 7')

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/content/models/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
YOLO: 80 COCO classes


Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/312M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

  VRAM  7.27 / 15.6 GB   after CLIP+BLIP+SD+Piper+traductor


  0%|                                               | 0.00/176M [00:00<?, ?B/s]

rembg listo (modelo de recorte de fondo cacheado)

=== Checklist de requisitos obligatorios ===
✅ Requisito 1/3 -- Modelo de visión: CLIP + BLIP + Stable Diffusion cargados
✅ Requisito 2/3 -- Modelo de audio: Piper TTS (es_MX-claude-high) cargado
ℹ️  Requisito 3/3 -- App Gradio en vivo: se confirma al correr la Sección 7


## 3b — Modelo de conversión de voz (RVC), entrenado con la voz pública

Para no depender de una sola voz sintética, este proyecto reutiliza el mismo motor de
clonación de voz (Applio/RVC), entrenado sobre la voz pública, la alternativa a clonar la propia voz.

**Origen y licencia (cita obligatoria):** hablante 09334 del *Crowdsourced high-quality
Colombian Spanish speech data set* (OpenSLR SLR72), bajo licencia CC BY-SA 4.0.

> Adriana Guevara-Rukoz, Işın Demirşahin, Fei He, Shan-Hui Cathy Chu, Supheakmungkol Sarin,
> Knot Pipatsrisawat, Alexander Gutkin, Alena Butryna, Oddur Kjartansson. "Crowdsourcing Latin
> American Spanish for Low-Resource Text-to-Speech". *LREC 2020*, pp. 6504–6513.

**Nota ética**: esta voz viene de un dataset público hecho
para investigación en síntesis de voz. Se usa aquí dentro del curso, tal como lo indica su
licencia — no se presenta como una persona real hablando en vivo.

El entrenamiento se hace **una sola vez**, aquí, junto con la carga de los demás modelos — no
en cada clic de "Generate". Después de esto, `narrate_caption()` primero sintetiza con Piper y luego convierte ese audio a esta voz con `rvc_convert()`.

No es obligatorio correr este paso ya que tarda varios minutos, se reocmienda colocar `USE_VOICE_CLONING = False` abajo, el resto de la app sigue funcionando igual, solo con la voz de Piper sin convertir.

In [5]:
# ACTIVAR/DESACTIVAR CLONACIÓN DE VOZ
USE_VOICE_CLONING = True   # False = solo Piper, sin este paso extra (mas simple y rapido)

if USE_VOICE_CLONING:
    # LIBRERÍAS
    import re, glob, shutil, subprocess, json
    import librosa
    import soundfile as sf

    # INSTALAR APPLIO (RVC)
    APPLIO = '/content/Applio'
    APY    = f'{APPLIO}/.venv/bin/python'
    APPLIO_SHA = '7b9f3fa0dde9f90946a5302b4ce4ab3410f12bb8'   # version de Applio con la que se probo este lab

    APPLIO_ENV = {k: v for k, v in os.environ.items() if k not in ('MPLBACKEND', 'PYTHONPATH', 'PYTHONHOME')}
    APPLIO_ENV['MPLBACKEND'] = 'Agg'
    APPLIO_ENV['PYTHONUNBUFFERED'] = '1'

    def applio(args, show=400):
        t0 = time.time()
        p = subprocess.run(f'{APY} core.py {args}', shell=True, cwd=APPLIO, env=APPLIO_ENV,
                           capture_output=True, text=True)
        out = p.stdout + p.stderr
        if p.returncode != 0 or 'An error occurred' in out:
            print(out[-3000:])
            raise RuntimeError(f'Applio `{args.split()[0]}` fallo')
        if show:
            print(out[-show:])
        print(f'--- core.py {args.split()[0]}: {time.time() - t0:.0f} s')
        return out

    t0 = time.time()
    if not os.path.exists(APY):
        !pip install -q uv
        !rm -rf {APPLIO} && mkdir -p {APPLIO}
        !curl -sL https://github.com/IAHispano/Applio/archive/{APPLIO_SHA}.tar.gz | tar -xz -C {APPLIO} --strip-components=1
        !uv venv -q {APPLIO}/.venv --python 3.12
        !uv pip install -q --python {APY} python-ffmpeg -r {APPLIO}/requirements.txt --extra-index-url https://download.pytorch.org/whl/cu128 --index-strategy unsafe-best-match
    applio('prerequisites --pretraineds-hifigan --models --no-exe', show=0)   # modelos base: 1.9 GB
    cfg = json.load(open(f'{APPLIO}/assets/config_template.json', encoding='utf-8'))
    cfg['precision'] = 'fp16'
    json.dump(cfg, open(f'{APPLIO}/assets/config.json', 'w', encoding='utf-8'), indent=2)
    print(f'Applio listo en {time.time() - t0:.0f} s')

    # VOZ PÚBLICA (dataset de L22)
    VOZ_DIR = f'{DRIVE_ROOT}/voz_publica'
    os.makedirs(VOZ_DIR, exist_ok=True)
    CANDIDATES = [
        f'{VOZ_DIR}/voz_publica_cof_09334_5min.wav',
        '/content/drive/MyDrive/TAE_IA_M6/inputs/voz/voz_publica_cof_09334_5min.wav',  # si ya la usaste en L22
    ]
    VOZ_PUBLICA_WAV = next((p for p in CANDIDATES if os.path.exists(p)), None)
    if VOZ_PUBLICA_WAV is None:
        print('No encontre voz_publica_cof_09334_5min.wav -- sube el .wav o el voz_publica.zip:')
        from google.colab import files
        for name, data in files.upload().items():
            dest = os.path.join(VOZ_DIR, name)
            with open(dest, 'wb') as f:
                f.write(data)
            if name.lower().endswith('.zip'):
                import zipfile
                with zipfile.ZipFile(dest) as z:
                    z.extractall(VOZ_DIR)
        VOZ_PUBLICA_WAV = next((p for p in [f'{VOZ_DIR}/voz_publica_cof_09334_5min.wav']
                               + glob.glob(f'{VOZ_DIR}/**/*.wav', recursive=True) if os.path.exists(p)), None)
        if VOZ_PUBLICA_WAV is None:
            raise SystemExit('No encontre el .wav de voz_publica despues de subirlo -- revisa el nombre del archivo.')
    print('Usando:', VOZ_PUBLICA_WAV)

    # PREPARAR DATASET DE ENTRENAMIENTO
    RVC_SR = 40000
    RVC_WORK = '/content/l12_rvc'
    DATASET = f'{RVC_WORK}/dataset'
    shutil.rmtree(DATASET, ignore_errors=True); os.makedirs(DATASET)

    y, _ = librosa.load(VOZ_PUBLICA_WAV, sr=RVC_SR, mono=True)
    y, _ = librosa.effects.trim(y, top_db=40)
    VOICE_WAV = f'{DATASET}/voice.wav'
    sf.write(VOICE_WAV, y, RVC_SR)
    print(f'Voz publica: {len(y) / RVC_SR / 60:.1f} min')

    def median_f0(path, seconds=30):
        yy, sr = librosa.load(path, sr=16000, mono=True, duration=seconds)
        f0, _, _ = librosa.pyin(yy, fmin=65, fmax=1000, sr=sr)
        return float(np.nanmedian(f0))

    F0_VOZ_PUBLICA = median_f0(VOICE_WAV)
    print(f'F0 mediana de la voz publica: {F0_VOZ_PUBLICA:.0f} Hz')

    # ENTRENAR MODELO DE VOZ
    RVC_NAME = 'voz_publica'
    shutil.rmtree(f'{APPLIO}/logs/{RVC_NAME}', ignore_errors=True)
    applio(f'preprocess --model-name {RVC_NAME} --dataset-path {DATASET} --sample-rate {RVC_SR} --cpu-cores 2', show=200)
    applio(f'extract --model-name {RVC_NAME} --f0-method rmvpe --sample-rate {RVC_SR} --cpu-cores 2 --gpu 0', show=200)

    RVC_EPOCHS, RVC_SAVE_EVERY, RVC_BATCH = 60, 20, 8
    RVC_TRAIN_LOG = f'{RVC_WORK}/train.log'
    rvc_train_proc = subprocess.Popen(
        f'{APY} core.py train --model-name {RVC_NAME} --save-every-epoch {RVC_SAVE_EVERY} --total-epoch {RVC_EPOCHS} '
        f'--sample-rate {RVC_SR} --batch-size {RVC_BATCH} --gpu 0',
        shell=True, cwd=APPLIO, env=APPLIO_ENV, stdout=open(RVC_TRAIN_LOG, 'w'), stderr=subprocess.STDOUT)
    RVC_TRAIN_T0 = time.time()
    print(f'Entrenamiento de voz iniciado en segundo plano (pid {rvc_train_proc.pid}). '
          f'Revisa el avance con rvc_training_status().')

    def rvc_training_status():
        log = open(RVC_TRAIN_LOG, encoding='utf-8', errors='ignore').read()
        if 'An error occurred' in log:
            print(log[-3000:]); raise RuntimeError('Applio reporto un error al entrenar la voz.')
        epochs = [int(e) for e in re.findall(r'epoch=(\d+)', log)]
        running = rvc_train_proc.poll() is None
        if not running and rvc_train_proc.returncode != 0:
            print(log[-3000:]); raise RuntimeError(f'Entrenamiento detenido con codigo {rvc_train_proc.returncode}.')
        ep, el = (max(epochs) if epochs else 0), time.time() - RVC_TRAIN_T0
        left = f'~{el / ep * (RVC_EPOCHS - ep) / 60:.0f} min restantes' if ep and running else ''
        print(f"{'entrenando' if running else 'LISTO'} | epoch {ep}/{RVC_EPOCHS} | {el/60:.1f} min | {left}")
        return running

    def rvc_wait_for_training(every=30):
        while rvc_training_status():
            time.sleep(every)

    rvc_training_status()
else:
    print('USE_VOICE_CLONING = False -- narrate_caption() usara solo la voz de Piper.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.4/20.4 MB 90.1 MB/s eta 0:00:00
--- core.py prerequisites: 68 s
Applio listo en 193 s
No encontre voz_publica_cof_09334_5min.wav -- sube el .wav o el voz_publica.zip:


Saving voz_publica.zip to voz_publica.zip
Usando: //content/drive/MyDrive/Colab Notebooks/Module 6/Dr. Francisco J. Rodriguez/voz_publica/voz_publica/voz_publica_cof_09334_5min.wav
Voz publica: 5.0 min
F0 mediana de la voz publica: 222 Hz
04:57 seconds of audio.
Model voz_publica preprocessed successfully.

100%|██████████| 1/1 [00:21<00:00, 21.37s/it]

--- core.py preprocess: 41 s
38it/s]
100%|██████████| 99/99 [00:05<00:00, 17.20it/s]

--- core.py extract: 47 s
Entrenamiento de voz iniciado en segundo plano (pid 7002). Revisa el avance con rvc_training_status().
entrenando | epoch 0/60 | 0.0 min | 


Se puede ejecutar la celda de abajo **después** de que `rvc_training_status()` haya impreso `LISTO` (no bloquea el resto del notebook, así que puedes seguir con la Sección 4 mientras entrena).

In [6]:
if USE_VOICE_CLONING:
    # ESPERAR A QUE TERMINE EL ENTRENAMIENTO
    rvc_wait_for_training()

    # CARGAR EL MODELO ENTRENADO
    RVC_CHECKPOINTS = sorted(glob.glob(f'{APPLIO}/logs/{RVC_NAME}/{RVC_NAME}_*e_*s.pth'),
                             key=lambda p: int(re.search(r'_(\d+)e_', p).group(1)))
    if not RVC_CHECKPOINTS:
        raise RuntimeError('Termino el entrenamiento pero no se guardo ningun modelo.')
    RVC_INDEX = (glob.glob(f'{APPLIO}/logs/{RVC_NAME}/*.index') or [None])[0]
    if RVC_INDEX is None:
        applio(f'index --model-name {RVC_NAME}', show=0)
        RVC_INDEX = glob.glob(f'{APPLIO}/logs/{RVC_NAME}/*.index')[0]
    RVC_FINAL = RVC_CHECKPOINTS[-1]
    print('Modelo de voz listo:', os.path.basename(RVC_FINAL), '|', os.path.basename(RVC_INDEX))

    # CONVERTIR AUDIO A LA VOZ ENTRENADA
    def rvc_convert(src_wav, out_wav, index_rate=0.5, protect=0.33):
        """Cualquier .wav hablado -> esa misma voz, convertida a la voz publica de
        L22 (RVC). El pitch se ajusta comparando la F0 mediana del audio de
        entrada contra la F0 mediana de la voz publica -- mismo calculo que L22.
        Esta conversion (a diferencia del entrenamiento) es rapida: segundos,
        no minutos, asi que si corre bien aqui en la app sin problema."""
        f0_src = median_f0(src_wav)
        pitch = int(np.clip(round(12 * np.log2(F0_VOZ_PUBLICA / f0_src)), -12, 12))
        applio(f'infer --input-path "{src_wav}" --output-path "{out_wav}" --pth-path "{RVC_FINAL}" '
              f'--index-path "{RVC_INDEX}" --pitch {pitch} --f0-method rmvpe '
              f'--index-rate {index_rate} --protect {protect}', show=0)
        if not os.path.exists(out_wav):
            raise RuntimeError(f'Applio no genero el archivo: {out_wav}')
        return out_wav

    print('Nota: este entrenamiento corre en un proceso/entorno de Python separado del '
          'principal, asi que no aparece en vram() -- eso solo mide CLIP+BLIP+SD+Piper.')

entrenando | epoch 0/60 | 0.0 min | 
entrenando | epoch 0/60 | 0.5 min | 
entrenando | epoch 0/60 | 1.0 min | 
entrenando | epoch 3/60 | 1.5 min | ~29 min restantes
entrenando | epoch 7/60 | 2.0 min | ~15 min restantes
entrenando | epoch 10/60 | 2.5 min | ~13 min restantes
entrenando | epoch 14/60 | 3.0 min | ~10 min restantes
entrenando | epoch 18/60 | 3.5 min | ~8 min restantes
entrenando | epoch 20/60 | 4.0 min | ~8 min restantes
entrenando | epoch 24/60 | 4.5 min | ~7 min restantes
entrenando | epoch 27/60 | 5.0 min | ~6 min restantes
entrenando | epoch 31/60 | 5.5 min | ~5 min restantes
entrenando | epoch 35/60 | 6.0 min | ~4 min restantes
entrenando | epoch 39/60 | 6.5 min | ~4 min restantes
entrenando | epoch 42/60 | 7.0 min | ~3 min restantes
entrenando | epoch 46/60 | 7.5 min | ~2 min restantes
entrenando | epoch 49/60 | 8.0 min | ~2 min restantes
entrenando | epoch 53/60 | 8.5 min | ~1 min restantes
entrenando | epoch 57/60 | 9.0 min | ~0 min restantes
entrenando | epoch 60/6

## 4 — The input contract, plus classification and a privacy guardrail

`coerce_image` carries over from L11 almost unchanged, with one addition:
`strip_exif()` runs on every upload, before anything else — see Section 8 for why.
`classify_category` is new: CLIP zero-shot over a fixed list of product categories,
instead of YOLO's fixed 80 COCO classes.

In [7]:
# LÍMITES DE TAMAÑO DE IMAGEN
MAX_SIDE, MIN_SIDE = 1024, 32

# CATEGORÍAS DE PRODUCTO (prompts para CLIP, en inglés)
PRODUCT_CATEGORIES = [
    'a t-shirt or clothing item',
    'a mug or cup',
    'a necklace',
    'a ring',
    'a bracelet',
    'other handmade jewelry, like earrings or anklets',
    'a candle',
    'a bag or purse',
    'a food or beverage product',
    'a piece of artwork or print',
    'a pair of shoes',
    'a skincare or beauty product',
    'a home decor item',
    'a toy or accessory',
    'other handmade product',
]

# TRADUCCIÓN DE CATEGORÍAS (lo que ve la usuaria)
CATEGORY_ES = {
    'a t-shirt or clothing item': 'una playera o prenda de ropa',
    'a mug or cup': 'una taza',
    'a necklace': 'un collar',
    'a ring': 'un anillo',
    'a bracelet': 'una pulsera',
    'other handmade jewelry, like earrings or anklets': 'otra joyería artesanal (aretes, tobilleras, etc.)',
    'a candle': 'una vela',
    'a bag or purse': 'una bolsa',
    'a food or beverage product': 'un producto alimenticio o bebida',
    'a piece of artwork or print': 'una obra de arte o impresión',
    'a pair of shoes': 'un par de zapatos',
    'a skincare or beauty product': 'un producto de belleza o cuidado de la piel',
    'a home decor item': 'un artículo de decoración para el hogar',
    'a toy or accessory': 'un juguete o accesorio',
    'other handmade product': 'otro producto artesanal',
}

# GUARDRAIL DE PRIVACIDAD (EXIF/GPS)
def strip_exif(img):
    """Return a copy of img with all EXIF metadata removed (GPS included).

    ETHICS GUARDRAIL. A phone photo's EXIF can carry exact GPS coordinates.
    Once this app's Gradio link is public, that metadata would travel with
    any image we display or save, silently leaking where the photo was
    taken -- so this runs on every upload, before anything else touches
    the image. Does NOT cover: location visible in the pixels themselves
    (a street sign, a house number, a distinctive background) -- only the
    hidden file metadata.
    """
    data = list(img.getdata())
    clean = Image.new(img.mode, img.size)
    clean.putdata(data)
    return clean

# CONTRATO DE ENTRADA (valida y normaliza cualquier imagen)
def coerce_image(x):
    """Anything a user can hand us -> PIL RGB, or ValueError with a readable message."""
    if x is None:
        raise ValueError('No image received. Please upload one.')
    if isinstance(x, np.ndarray):            # Gradio hands you numpy, not PIL
        x = Image.fromarray(x.astype(np.uint8))
    if not isinstance(x, Image.Image):
        raise ValueError('Unsupported input. Please upload an image file.')
    x = strip_exif(x)                        # guardrail: drop GPS/EXIF before anything else
    x = x.convert('RGB')                     # handles L, RGBA, P, CMYK
    w, h = x.size
    if min(w, h) < MIN_SIDE:
        raise ValueError(f'Image too small ({w}x{h}). Use at least {MIN_SIDE}x{MIN_SIDE}.')
    if max(w, h) > MAX_SIDE:                 # a 12 MP phone photo OOMs a T4
        s = MAX_SIDE / max(w, h)
        x = x.resize((int(w * s), int(h * s)), Image.LANCZOS)
    return x

# DETECCIÓN DE OBJETOS (YOLO)
def detect(img, conf=0.25):
    """PIL -> [(label, confidence, (x1,y1,x2,y2))], strongest first."""
    res = yolo(np.array(img), verbose=False)
    out = [(res[0].names[int(b.cls)], float(b.conf), tuple(map(int, b.xyxy[0])))
           for b in res[0].boxes if float(b.conf) >= conf]
    return sorted(out, key=lambda b: -b[1])

# CLASIFICACIÓN DE CATEGORÍA (CLIP)
def classify_category(img, categories=PRODUCT_CATEGORIES):
    """PIL -> (categoria_en_espanol, prob). Zero-shot product classification
    with CLIP. CLIP clasifica con los prompts en ingles (arriba) -- traducir
    esos prompts bajaria bastante la precision, ya que el modelo casi no vio
    texto en espanol al entrenarse -- y aqui solo se traduce la ETIQUETA que
    se devuelve, usando CATEGORY_ES."""
    inputs = clip_proc(
        text=[f'a photo of {c}' for c in categories],
        images=img, return_tensors='pt', padding=True
    ).to('cuda')
    with torch.no_grad():
        logits = clip_model(**inputs).logits_per_image
    probs = logits.softmax(dim=1).squeeze().tolist()
    ranked = sorted(zip(categories, probs), key=lambda p: -p[1])
    top_label_en, top_prob = ranked[0]
    return CATEGORY_ES[top_label_en], top_prob

# DESCRIPCIÓN DE LA IMAGEN (BLIP)
def caption(img, max_new_tokens=40):
    """PIL -> str. float16 in must match how the model was loaded."""
    inp = blip_proc(images=img.convert('RGB'), return_tensors='pt').to('cuda', torch.float16)
    with torch.no_grad():
        out = blip_model.generate(**inp, max_new_tokens=max_new_tokens, num_beams=3)
    return blip_proc.decode(out[0], skip_special_tokens=True).strip()

## 5 — The pipeline

Two stages run in parallel conceptually but independently of each other — **CLIP**
(category) and **BLIP** (caption) — then both feed into a **third** stage,
Stable Diffusion, which turns the caption into a lifestyle-photography prompt and
generates several styled variants of the product photo. Each style now has a name and
its own personality (including a joke "meme studio" option), and a `surprise` flag can
pick styles at random instead of taking the first N.

`run_product_pipeline` is the only function the interface calls, and — like L11's
`run_pipeline` — it never raises.

In [8]:
# ESTILOS DISPONIBLES
LIFESTYLE_STYLES = {
    'Fondo negro profesional':   None,
    'Atardecer en la playa':     'on a sandy beach at golden hour, warm sunset light, soft shadows',
    'Loft minimalista':          'in a minimalist loft, soft natural light, clean neutral background',
    'Joyería en busto (fondo negro dramático)': None,
    'Mercado vintage':           'on a rustic wooden market stall, vintage props, warm ambient light',
    'Neón nocturno':             'in a neon-lit night scene, vibrant colors, reflective urban backdrop',
    'Estudio meme (modo broma)': 'on a messy cluttered desk, harsh direct flash, deliberately awkward amateur photo',
}

# NEGATIVE PROMPT
NEGATIVE_PROMPT = 'blurry, low quality, distorted, watermark, text, extra objects, deformed'

# PLANTILLAS DE COPY PARA REDES
SOCIAL_CAPTION_TEMPLATES = [
    "✨ Conoce nuestro/a {category} ✨\n{caption}\n#HechoALaMano #ShopSmall",
    "📸 {caption}\n¡Ya disponible! Escríbenos para el tuyo 💬\n#EmprendimientoLocal",
    "Cuando tu {category} se ve así de bien... 😍\n#ProductoDelDía #MadeWithLove",
]

def make_social_caption(category, caption_text):
    """Plantilla simple para un pie de foto listo para redes.
    La descripcion de BLIP viene en ingles, se muestra tal cual, sin traducir"""
    template = random.choice(SOCIAL_CAPTION_TEMPLATES)
    return template.format(category=category, caption=caption_text)

# TRADUCCIÓN PARA NARRACIÓN
def translate_caption(text_en):
    """Descripcion de BLIP (ingles) -> version en espanol, solo para narrar en voz"""
    try:
        batch = mt_tok([text_en], return_tensors='pt', padding=True).to('cuda')
        with torch.no_grad():
            out = mt_model.generate(**batch, max_new_tokens=60)
        return mt_tok.decode(out[0], skip_special_tokens=True).strip()
    except Exception as e:
        print(f'[translate_caption] {type(e).__name__}: {e}')
        return None

# GUION DE NARRACIÓN
NARRATION_TEMPLATES = [
    '¿Buscas algo especial? Aquí tienes {category}: {caption_es}. Escríbenos hoy '
    'mismo. Imagen conceptual creada con inteligencia artificial.',
    'Presentamos {category}: {caption_es}. Hecho a mano, con dedicación. '
    'Contáctanos para más información. Foto generada con inteligencia artificial.',
    'Esto te va a encantar: {category}, {caption_es}. Escríbenos para más '
    'detalles. Imagen creada con inteligencia artificial.',
]

# RESPALDO SI NO HAY TRADUCCIÓN DISPONIBLE
NARRATION_FALLBACK_TEMPLATES = [
    '¿Buscas algo especial? Mira {category}, disponible ahora. Imagen '
    'conceptual creada con inteligencia artificial.',
    'Presentamos {category}, hecho a mano con dedicación. Escríbenos para más '
    'información.',
    'Esto te va a encantar: {category}. Contáctanos hoy mismo. Foto generada '
    'con inteligencia artificial.',
]

def make_narration_text(category, caption_es=None):
    """Categoria (español) + descripcion de BLIP ya traducida -> guion corto de
    voz tipo comercial. Si no hay traduccion disponible, usa solo la categoria."""
    if caption_es:
        return random.choice(NARRATION_TEMPLATES).format(category=category, caption_es=caption_es)
    return random.choice(NARRATION_FALLBACK_TEMPLATES).format(category=category)

# AJUSTES DE VOZ (Piper) - length_scale, noise_scale, noise_w_scale
AD_VOICE_CONFIG = SynthesisConfig(length_scale=1.12, noise_scale=0.5, noise_w_scale=0.6)

import re

# LIMPIEZA DE TEXTO PARA TTS (quita emoji y hashtags antes de sintetizar)
_EMOJI_RE = re.compile(
    '[\U0001F300-\U0001FAFF\U00002600-\U000027BF\U0001F1E6-\U0001F1FF]+'
)


def _clean_for_tts(text):
    """Strip emoji and #hashtags before synthesis. Everything else (the actual
    caption sentence) is kept as-is."""
    text = _EMOJI_RE.sub('', text)
    text = re.sub(r'#\w+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def narrate_caption(text, out_path):
    """Social caption text -> narrated .wav at out_path, or None on failure.
    Two audio models in sequence: Piper (Section 3) synthesizes the raw
    speech, then, if USE_VOICE_CLONING is on, the RVC voice model (Section 3b,
    trained on L22's public voice) converts it to that voice. If the RVC step
    fails for any reason -- or was turned off -- falls back to the plain
    Piper narration instead of losing the audio output. Never raises, same
    contract as the rest of this pipeline."""
    clean = _clean_for_tts(text)
    if not clean:
        return None
    piper_path = out_path.replace('.wav', '_piper.wav')
    try:
        with wave.open(piper_path, 'wb') as wf:
            piper_voice.synthesize_wav(clean, wf, syn_config=AD_VOICE_CONFIG)
    except Exception as e:
        print(f'[narrate_caption] {type(e).__name__}: {e}')
        return None
    # globals().get(...): evita un NameError si la Seccion 3b no corrio en esta sesion
    if globals().get('USE_VOICE_CLONING') and 'rvc_convert' in globals():
        try:
            return rvc_convert(piper_path, out_path)
        except Exception as e:
            print(f'[narrate_caption] la conversion RVC fallo, uso la voz de Piper tal cual: '
                  f'{type(e).__name__}: {e}')
            return piper_path
    return piper_path


# ANÁLISIS DEL PRODUCTO (CLIP + BLIP + traducción)
def analyse_product(img):
    t = {}
    t0 = time.time(); category, cat_prob = classify_category(img); t['clip'] = time.time() - t0
    t0 = time.time(); text = caption(img); t['blip'] = time.time() - t0
    t0 = time.time(); text_es = translate_caption(text); t['translate'] = time.time() - t0
    return {'category': category, 'cat_prob': cat_prob, 'caption': text,
            'caption_es': text_es, 'timings': t}

# ESTILO "FONDO NEGRO PROFESIONAL" (inpainting)
PRO_BG_PROMPT = ('professional product photography, on a black leather '
                 'textured background, dramatic soft studio lighting, elegant '
                 'rim light, high-end catalog photo, subtle reflections')


def make_pro_black_background(img, seed=0, steps=25, guidance_scale=7.5):
    """Foto de producto tipo catalogo (fondo de cuero negro con textura,
    iluminacion de estudio). Composita: SD SOLO genera el fondo; el producto
    se pega tal cual (pixeles originales) para que quede 100% fiel a la foto,
    sin distorsion en los bordes."""
    square = img.convert('RGB').resize((512, 512))
    cutout = rembg_remove(square)
    alpha = cutout.split()[-1]

    # máscara para SD: blanco = fondo (que lo invente), negro = producto
    # (que NO lo toque) -- sin erosionar ni difuminar, para no cederle ni un
    # pixel del producto a la difusión.
    mask = alpha.point(lambda a: 0 if a > 200 else 255)

    background = pipe_sd_inpaint(
        prompt=PRO_BG_PROMPT, negative_prompt=NEGATIVE_PROMPT,
        image=square, mask_image=mask,
        guidance_scale=guidance_scale, num_inference_steps=steps,
        generator=torch.Generator('cuda').manual_seed(seed)).images[0]

    # pega el producto ORIGINAL sobre el fondo generado (alpha de rembg ya
    # trae el borde suavizado, así que el pegado queda limpio)
    result = background.convert('RGB')
    result.paste(square, mask=alpha)
    return result


# ESTILO "JOYERÍA EN BUSTO" (inpainting) -- NUEVO
JEWELRY_BUST_PROMPT = ('elegant jewelry chain draped on a black velvet display '
                       'bust, dramatic side rim lighting, deep rich shadows, '
                       'luxury jewelry catalog photography, subtle sparkle and '
                       'reflections on the metal, professional studio setup')


def make_jewelry_bust_style(img, seed=0, steps=25, guidance_scale=7.5):
    """Foto de joyeria tipo catalogo (collar/cadena sobre busto de terciopelo
    negro, iluminacion dramatica) -- misma tecnica de composicion que
    make_pro_black_background(): SD solo genera el fondo, la cadenita se pega
    tal cual (100% fiel, sin distorsion en los eslabones)."""
    square = img.convert('RGB').resize((512, 512))
    cutout = rembg_remove(square)
    alpha = cutout.split()[-1]

    mask = alpha.point(lambda a: 0 if a > 200 else 255)

    background = pipe_sd_inpaint(
        prompt=JEWELRY_BUST_PROMPT, negative_prompt=NEGATIVE_PROMPT,
        image=square, mask_image=mask,
        guidance_scale=guidance_scale, num_inference_steps=steps,
        generator=torch.Generator('cuda').manual_seed(seed)).images[0]

    result = background.convert('RGB')
    result.paste(square, mask=alpha)
    return result


# GENERACIÓN DE VARIANTES "LIFESTYLE"
def generate_lifestyle(img, text, styles, strength=0.6, steps=20, seed=SEED):
    """styles: lista de nombres (llaves de LIFESTYLE_STYLES), una imagen por
    estilo. Si LIFESTYLE_STYLES[style] es None (los estilos de fondo negro), usa
    pipe_sd_inpaint en vez de pipe_sd -- ver make_pro_black_background() y
    make_jewelry_bust_style() arriba."""
    outs = []
    for i, style_name in enumerate(styles):
        style_prompt = LIFESTYLE_STYLES[style_name]
        if style_prompt is None:
            if style_name == 'Joyería en busto (fondo negro dramático)':
                out = make_jewelry_bust_style(img, seed=seed + i)
                prompt = f'[inpainting] {JEWELRY_BUST_PROMPT}'
            else:
                out = make_pro_black_background(img, seed=seed + i)
                prompt = f'[inpainting] {PRO_BG_PROMPT}'
        else:
            prompt = f'{text}, {style_prompt}'
            out = pipe_sd(prompt=prompt, negative_prompt=NEGATIVE_PROMPT,
                          image=img.resize((512, 512)), strength=strength,
                          guidance_scale=7.5, num_inference_steps=steps,
                          generator=torch.Generator('cuda').manual_seed(seed + i)).images[0]
        outs.append((out, style_name, prompt))
    return outs

# PIPELINE COMPLETO (une todo lo de arriba)
def run_product_pipeline(x, strength=0.6, n_variants=3, surprise=False):
    """Anything in, never raises. Always returns (list_of_images, message, audio_path)."""
    try:
        img = coerce_image(x)
    except ValueError as e:
        return [], str(e), None # the user's fault: show them the sentence

    style_names = list(LIFESTYLE_STYLES)
    n_variants = max(1, min(n_variants, len(style_names)))
    chosen = random.sample(style_names, n_variants) if surprise else style_names[:n_variants]

    try:
        a = analyse_product(img)
        variants = generate_lifestyle(img, a['caption'], styles=chosen, strength=strength)
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        return [], 'The GPU ran out of memory. Try a smaller image.', None
    except Exception as e:
        print(f'[run_product_pipeline] {type(e).__name__}: {e}')
        return [], 'Something went wrong. Try another image.', None

    images_only    = [v[0] for v in variants]
    style_list     = ', '.join(v[1] for v in variants)
    social_caption = make_social_caption(a['category'], a['caption'])

    # requisito de audio: narra en español (ya traducida) la descripción de BLIP;
    # nunca detiene el reporte si falla, cae a una frase con solo la categoría
    audio_path = narrate_caption(make_narration_text(a['category'], a['caption_es']),
                                 os.path.join(OUTPUT_DIR, 'narracion.wav'))
    narration_note = ('lista, reprodúcela abajo' if audio_path else
                      'no se pudo generar esta vez (revisa tu conexión)')

    report = (f"category    : {a['category']} ({a['cat_prob']:.1%})\n"
              f"caption (EN): {a['caption']}\n"
              f"caption (ES): {a['caption_es'] or '(traducción no disponible)'}\n"
              f"styles      : {style_list}\n"
              f"time        : clip {a['timings']['clip']*1000:.0f} ms, blip {a['timings']['blip']:.2f} s, "
              f"traducción {a['timings']['translate']*1000:.0f} ms\n"
              f"variants    : {len(variants)}\n\n"
              f"\U0001F916 Generado con IA -- no es la foto original\n"
              f"Sugerencia de copy: {social_caption}\n"
              f"\U0001F50A Narración: {narration_note}")
    return images_only, report, audio_path

## 6 — Test it before it ever meets a seller

Five product photos, five categories to predict correctly. The hostile-input block
matters just as much here: this app has a public link too.

In [9]:
# PRUEBA: CATEGORÍA + DESCRIPCIÓN EN LAS FOTOS DE ENTRADA
imgs = load_inputs()
print(f'--- category + caption on {len(imgs)} images ---')
for name, im in imgs.items():
    category, prob = classify_category(im)
    text = caption(im)
    print(f'{name[:20]:20s} -> {category} ({prob:.1%}) | {text}')

# PRUEBA: ENTRADAS HOSTILES
ref = next(iter(imgs.values()))
print('\n--- hostile inputs: none of these may raise ---')
for tag, bad in [('None', None),
                 ('numpy', np.zeros((64, 64, 3), dtype=np.uint8)),
                 ('grayscale', ref.convert('L')),
                 ('RGBA', ref.convert('RGBA')),
                 ('1x1', Image.new('RGB', (1, 1))),
                 ('12 MP', Image.new('RGB', (4000, 3000), 'white'))]:
    _imgs, msg, _aud = run_product_pipeline(bad, n_variants=1)
    print(f'  {tag:10s} -> {msg.splitlines()[0][:60]}')

# ELEGIR LA FOTO PARA LA CORRIDA REAL (nombre tal cual aparece arriba o en
# L12_inputs/; deja None para tomar la primera en orden alfabético)
FOTO_ELEGIDA = 'cadena.jpg'   # <- cambiar esto por el archivo que se desea

if FOTO_ELEGIDA and FOTO_ELEGIDA in imgs:
    best = FOTO_ELEGIDA
elif FOTO_ELEGIDA:
    raise SystemExit(f'"{FOTO_ELEGIDA}" no esta en L12_inputs/. Opciones: {list(imgs)}')
else:
    best = next(iter(imgs))

# CORRIDA REAL
print(f'\n--- real run: {best} ---')
variants, report, audio_path = run_product_pipeline(imgs[best], n_variants=3)
print(report); vram('peak')
if audio_path:
    from IPython.display import Audio, display
    print('narration ->', audio_path)
    display(Audio(audio_path))

# GUARDAR RESULTADOS EN DRIVE
for i, v in enumerate(variants):
    out_path = os.path.join(OUTPUT_DIR, f'{os.path.splitext(best)[0]}_estilo_{i}.jpg')
    v.save(out_path)
    print('saved ->', out_path)

--- category + caption on 4 images ---
Collar.jpeg          -> un collar (98.1%) | there is a gold heart pendant on a gold chain on a white blanket
cadena.jpg           -> un collar (97.5%) | a close up of a silver chain on a white background
cup.jpg              -> una taza (98.6%) | there is a white cup with a gold handle sitting on a table
t-shirt.jpg          -> una playera o prenda de ropa (99.8%) | there is a white t - shirt hanging on a hanger against a concrete wall

--- hostile inputs: none of these may raise ---
  None       -> No image received. Please upload one.
--- core.py infer: 19 s
  numpy      -> category    : un producto alimenticio o bebida (28.5%)
--- core.py infer: 14 s
  grayscale  -> category    : un collar (92.7%)
--- core.py infer: 13 s
  RGBA       -> category    : un collar (98.1%)
  1x1        -> Image too small (1x1). Use at least 32x32.
--- core.py infer: 14 s
  12 MP      -> category    : una playera o prenda de ropa (23.4%)

--- real run: cadena.jpg ---

saved -> //content/drive/MyDrive/Colab Notebooks/Module 6/Dr. Francisco J. Rodriguez/L12_output/cadena_estilo_0.jpg
saved -> //content/drive/MyDrive/Colab Notebooks/Module 6/Dr. Francisco J. Rodriguez/L12_output/cadena_estilo_1.jpg
saved -> //content/drive/MyDrive/Colab Notebooks/Module 6/Dr. Francisco J. Rodriguez/L12_output/cadena_estilo_2.jpg


## 7 — The app

Same `Blocks` pattern as L11, with two interface changes from a plain restyle app: the
output is a `gr.Gallery` of multiple generated variants instead of a single restyled
image, since the whole point here is giving a seller options to choose from — and a
"🎲 Sorpréndeme" checkbox lets the seller skip picking a style altogether. Each result also
carries a suggested social-media caption and a visible AI-generated badge.

In [10]:
# LIBRERÍAS
import gradio as gr

import gradio_client.utils as _gc_utils
_orig_json_schema_to_python_type = _gc_utils.json_schema_to_python_type

def _safe_json_schema_to_python_type(schema):
    try:
        return _orig_json_schema_to_python_type(schema)
    except Exception:
        return 'Any'

_gc_utils.json_schema_to_python_type = _safe_json_schema_to_python_type


# FUNCIÓN QUE CONECTA LA APP CON EL PIPELINE
def predict(image, strength, n_variants, surprise, progress=gr.Progress()):
    if image is None:
        return None, 'Please upload an image first.', None

    progress(0.3, desc='Analyzing product...')
    variants, report, audio_path = run_product_pipeline(
        image, strength=strength, n_variants=int(n_variants), surprise=surprise,
    )
    progress(1.0, desc='Done')
    return variants, report, audio_path


# INTERFAZ (Gradio Blocks)
with gr.Blocks(title='Product Photo Generator') as app:
    gr.Markdown('# Product Photo Generator\nUpload a photo of your product. It identifies the '
                'category, describes it, generates lifestyle-style marketing photos, and narrates '
                'the suggested caption out loud.\n\n'
                '_Uploaded photos are stripped of EXIF/GPS metadata before processing._')
    with gr.Row():
        with gr.Column():
            inp        = gr.Image(type='pil', label='Your product photo')
            strength   = gr.Slider(0.3, 0.9, value=0.6, step=0.05, label='Style strength')
            n_variants = gr.Slider(1, 3, value=3, step=1, label='Number of variants')
            surprise   = gr.Checkbox(label='🎲 Sorpréndeme con el estilo', value=False)
            with gr.Row():
                go    = gr.Button('Generate', variant='primary')
                clear = gr.Button('Clear')
            gr.Examples(examples=[[os.path.join(INPUT_DIR, f)] for f in load_inputs()],
                        inputs=inp)
        with gr.Column():
            out_gallery = gr.Gallery(label='Lifestyle versions', columns=3)
            out_report  = gr.Textbox(label='What happened', lines=10)
            out_audio   = gr.Audio(label='🔊 Narración del copy (texto a voz)', type='filepath')

    go.click(predict, [inp, strength, n_variants, surprise], [out_gallery, out_report, out_audio])
    clear.click(lambda: (None, False, None, None, None),
               None, [inp, surprise, out_gallery, out_report, out_audio])

print('▶️  Requisito 3/3 -- App Gradio en vivo: lanzando con share=True (link publico)...')
app.queue().launch(share=True, quiet=True, show_api=False)

▶️  Requisito 3/3 -- App Gradio en vivo: lanzando con share=True (link publico)...
* Running on public URL: https://d5002bdbe1c6bb2073.gradio.live


---
## 8 — Ethics guardrail

**Misuse:** a seller uploads a product photo taken with their phone at home to
generate marketing images. This app's Gradio link is public (`share=True`), so
the uploaded image -- and any hidden metadata in it -- is reachable by anyone
with the link, not just the seller.

**Guardrail:** `strip_exif()`, called inside `coerce_image()` on every upload,
before anything else touches the image. It rebuilds the image from pixel data
only, discarding all EXIF (GPS coordinates included).

**What it does not cover:** location or identity visible in the pixels
themselves -- a street sign, a house number, a distinctive room in the
background. `strip_exif()` only removes hidden file metadata, not what the
camera captured on the sensor.

In [11]:
# DEMOSTRACIÓN DEL GUARDRAIL (antes/después de strip_exif)
sample = next(iter(load_inputs().values()))
print('EXIF before strip_exif():', sample.getexif() if hasattr(sample, 'getexif') else 'n/a')

clean = strip_exif(sample)
print('EXIF after strip_exif() :', clean.getexif())

EXIF before strip_exif(): {}
EXIF after strip_exif() : {}


## 9 — Cleanup — Track A closes here

L01–L10 cached model weights **on Drive**, under `TAE_IA_M6/models/`. From L11 onward, models
live on the runtime disk instead — that folder is dead weight now, and on a free 15 GB account
it's the difference between Whisper downloading cleanly in L19 and Track B failing outright.

Read the size table first. Nothing is deleted until `CONFIRM = True` and the cell is re-run.

In [12]:
# LIBRERÍAS
import shutil

# CARPETA LEGADO Y HELPER DE TAMAÑO
LEGACY = f'{DRIVE_ROOT}/models'

def gb(path):
    return sum(os.path.getsize(os.path.join(d, f))
               for d, _, fs in os.walk(path) for f in fs) / 1e9

# REVISAR Y BORRAR EL CACHÉ LEGADO (solo si CONFIRM = True)
if os.path.exists(LEGACY):
    print(f'{LEGACY}  ({gb(LEGACY):.2f} GB)')
    for e in sorted(os.listdir(LEGACY)):
        print(f'  {gb(os.path.join(LEGACY, e)):6.2f} GB  {e}')
    CONFIRM = False                      # set True and re-run to delete
    if CONFIRM:
        freed = gb(LEGACY); shutil.rmtree(LEGACY)
        print(f'\nFreed {freed:.2f} GB.')
    else:
        print('\nReview the list, then set CONFIRM = True and re-run.')
else:
    print('Nothing to clean.')

# VERIFICAR QUE OUTPUTS/INPUTS SIGAN INTACTOS
print('\nOutputs kept:', os.path.exists(OUTPUT_DIR))
print('Inputs kept :', os.path.exists(INPUT_DIR),
      f'({len(os.listdir(INPUT_DIR))} files)' if os.path.exists(INPUT_DIR) else '')

//content/drive/MyDrive/Colab Notebooks/Module 6/Dr. Francisco J. Rodriguez/models  (20.48 GB)
   14.98 GB  blip2-opt-2.7b-local
    0.61 GB  clip-vit-base-patch32-local
    0.03 GB  fast_neural_style
    0.01 GB  gradio
    0.02 GB  hub
    0.00 GB  medium.pt
    2.74 GB  sd15-local
    0.00 GB  small.pt
    0.00 GB  tiny.pt
    0.00 GB  torch_home

Review the list, then set CONFIRM = True and re-run.

Outputs kept: True
Inputs kept : True (4 files)


---
*TAE-IA 2025 · Cocyten-Nayarit · Módulo 6 · L12*